# imports

In [ ]:
from preprocess import get_data
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split

In [ ]:
new_df = get_data()
new_df

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.2,
    random_state=42,
)

In [ ]:
def find_best_hyperparameters(
    regressor, param_grid, X_train, y_train, scoring="r2", sample_weights=None
):
    grid_search = GridSearchCV(
        regressor,
        param_grid,
        cv=5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1,
        verbose=0,
    )  # use all the cores of the CPU
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    print("Best hyperparameters: {}".format(grid_search.best_params_))
    print("Best score (r2): {:.2f}".format(grid_search.best_score_))
    return grid_search.best_estimator_

In [ ]:
def print_cross_validation_results(regressor, X_train, y_train):
    scoring = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]
    cv_results = cross_validate(
        regressor,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
        verbose=0,
        return_train_score=True,
    )
    print("Train R^2: {:.2f}".format(cv_results["train_r2"].mean()))
    print("Validation R^2: {:.2f}".format(cv_results["test_r2"].mean()))
    print(
        "Train MAE: {:.2f}".format(-cv_results["train_neg_mean_absolute_error"].mean())
    )
    print(
        "Validation MAE: {:.2f}".format(
            -cv_results["test_neg_mean_absolute_error"].mean()
        )
    )
    print(
        "Train RMSE: {:.2f}".format(
            -cv_results["train_neg_root_mean_squared_error"].mean()
        )
    )
    print(
        "Validation RMSE: {:.2f}".format(
            -cv_results["test_neg_root_mean_squared_error"].mean()
        )
    )

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.4,
    random_state=42,
)


param_grid = {
    "n_estimators": [10, 100, 1000],
    "learning_rate": [0.1, 0.05, 0.01, 0.001],
}

clf = GradientBoostingRegressor(
    n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0
)

clf = find_best_hyperparameters(
    clf,
    param_grid,
    X_train,
    y_train,
)
print_cross_validation_results(clf, X_train, y_train)